In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans

from jColor.Color import Color
from jColor.Image import Image

In [ ]:
SAMPLE = 'M1'
MAX_PIXELS = 100_000
RANDOM_STATE = 0

image_path = f'imgs/{SAMPLE}.jpg'
jade_image = Image(str(image_path))
normalized_rgb, white_reference, foreground_mask = jade_image.RGB_Normalization()
foreground = foreground_mask.astype(bool)
lab_pixels = Color(normalized_rgb).Lab[foreground]

rng = np.random.default_rng(RANDOM_STATE)
if len(lab_pixels) > MAX_PIXELS:
    selected_indices = rng.choice(len(lab_pixels), size=MAX_PIXELS, replace=False)
    elbow_data = lab_pixels[selected_indices]
else:
    elbow_data = lab_pixels.copy()

In [ ]:
K_VALUES = np.arange(1, 11)
inertias = []

for k in K_VALUES:
    model = KMeans(
        n_clusters=int(k),
        init='k-means++',
        n_init=10,
        random_state=RANDOM_STATE,
    )
    model.fit(elbow_data)
    inertias.append(model.inertia_)
    print(f'k={k:2d} | inertia={model.inertia_:,.2f}')

inertias = np.asarray(inertias)
relative_reduction = np.full(len(inertias), np.nan)
relative_reduction[1:] = 100 * (inertias[:-1] - inertias[1:]) / inertias[:-1]

elbow_table = pd.DataFrame({
    'k': K_VALUES,
    'inertia': inertias,
    'percentage_reduction': relative_reduction,
})

In [ ]:
x_normalized = (K_VALUES - K_VALUES.min()) / (K_VALUES.max() - K_VALUES.min())
y_normalized = (inertias - inertias.min()) / (inertias.max() - inertias.min())

start = np.array([x_normalized[0], y_normalized[0]])
end = np.array([x_normalized[-1], y_normalized[-1]])
points = np.column_stack((x_normalized, y_normalized))
line_vector = end - start
offsets = points - start
cross_magnitudes = np.abs(line_vector[0] * offsets[:, 1] - line_vector[1] * offsets[:, 0])
distances = cross_magnitudes / np.linalg.norm(line_vector)
suggested_k = int(K_VALUES[np.argmax(distances)])

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

axes[0].plot(K_VALUES, inertias, marker='o', color='darkblue')
axes[0].axvline(suggested_k, color='red', linestyle='--', label=f'Suggested elbow: k={suggested_k}')
axes[0].scatter(suggested_k, inertias[suggested_k - 1], color='red', s=80, zorder=3)
axes[0].set_xticks(K_VALUES)
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title(f'Elbow method — {SAMPLE}')
axes[0].grid(alpha=0.25)
axes[0].legend()

axes[1].bar(K_VALUES[1:], relative_reduction[1:], color='slateblue')
axes[1].set_xticks(K_VALUES[1:])
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Inertia reduction relative to k-1 (%)')
axes[1].set_title('Gain from adding each cluster')
axes[1].grid(axis='y', alpha=0.25)

plt.show()
print(f'Suggested value from geometric distance: k = {suggested_k}')